# Ladybug + Icebug: in-memory manual-insert Leiden

This notebook builds an in-memory Ladybug graph, exports it to Parquet `EXPORT DATABASE`, imports it into a fresh in-memory instance, extracts CSR with `query_as_arrow(...).csr()`, runs Icebug Leiden, writes the communities back onto the nodes, and queries the updated graph.


## Environment

Dependencies are declared in `pyproject.toml`. From a fresh checkout, start Jupyter with:

```bash
uv run jupyter notebook ladybug_mem_icebug_mem.ipynb
```

Or execute the notebook from the CLI with:

```bash
uv run jupyter nbconvert --to notebook --execute ladybug_mem_icebug_mem.ipynb --inplace
```

In [1]:
import os
import shutil
import tempfile
from pathlib import Path

import icebug as ib
import ladybug
import pandas as pd
import pyarrow as pa

print("ladybug", ladybug.__version__)
print("icebug", ib.__version__)

workdir = Path(tempfile.mkdtemp(prefix="ladybug_import_export_"))
export_dir = workdir / "exported-db"
print("workdir:", workdir)

ladybug 0.17.0
icebug 12.9
workdir: /tmp/ladybug_import_export_az6fdq0p


## Build an in-memory Ladybug graph

We construct a **high-modularity graph**: 4 complete cliques (K₅, 5 nodes each = 20 nodes total) connected by 3 minimal bridge edges.
- 40 internal edges (K₅ = C(5,2) = 10 edges per clique)
- 3 bridge edges (sparse inter-community connections)

In [2]:
source_db = ladybug.Database()
source = ladybug.Connection(source_db)

people = [
    {"id": 0, "name": "Ada", "team": "A", "community": 0},
    {"id": 1, "name": "Ben", "team": "A", "community": 0},
    {"id": 2, "name": "Cora", "team": "A", "community": 0},
    {"id": 3, "name": "Drew", "team": "A", "community": 0},
    {"id": 4, "name": "Eli", "team": "A", "community": 0},
    {"id": 5, "name": "Fay", "team": "B", "community": 0},
    {"id": 6, "name": "Gus", "team": "B", "community": 0},
    {"id": 7, "name": "Ivy", "team": "B", "community": 0},
    {"id": 8, "name": "Jin", "team": "B", "community": 0},
    {"id": 9, "name": "Kai", "team": "B", "community": 0},
    {"id": 10, "name": "Luz", "team": "C", "community": 0},
    {"id": 11, "name": "Moe", "team": "C", "community": 0},
    {"id": 12, "name": "Nia", "team": "C", "community": 0},
    {"id": 13, "name": "Oli", "team": "C", "community": 0},
    {"id": 14, "name": "Pia", "team": "C", "community": 0},
    {"id": 15, "name": "Quin", "team": "D", "community": 0},
    {"id": 16, "name": "Rae", "team": "D", "community": 0},
    {"id": 17, "name": "Sol", "team": "D", "community": 0},
    {"id": 18, "name": "Tia", "team": "D", "community": 0},
    {"id": 19, "name": "Uma", "team": "D", "community": 0},
]

# 4 cliques (K5): 40 internal undirected edges + 3 sparse bridges.
undirected_edges = []
for start in (0, 5, 10, 15):
    for i in range(start, start + 5):
        for j in range(i + 1, start + 5):
            undirected_edges.append((i, j, 10))

# Minimal inter-community bridges to keep components connected.
undirected_edges += [
    (0, 5, 1),
    (5, 10, 1),
    (10, 15, 1),
]

edge_rows = []
for src, dst, strength in undirected_edges:
    edge_rows.append({"src": src, "dst": dst, "strength": strength})
    edge_rows.append({"src": dst, "dst": src, "strength": strength})

def cypher_list(rows, keys):
    items = []
    for row in rows:
        fields = []
        for key in keys:
            value = row[key]
            if isinstance(value, str):
                fields.append(f"{key}: '{value}'")
            else:
                fields.append(f"{key}: {int(value)}")
        items.append("{" + ", ".join(fields) + "}")
    return "[" + ", ".join(items) + "]"

source.execute("""
CREATE NODE TABLE Person(
    id INT64,
    name STRING,
    team STRING,
    community INT64,
    PRIMARY KEY(id)
)
""")
source.execute("""
CREATE REL TABLE KNOWS(
    FROM Person TO Person,
    strength INT64
)
""")
source.execute(f"""
UNWIND {cypher_list(people, ['id', 'name', 'team', 'community'])} AS row
CREATE (:Person {{
    id: row.id,
    name: row.name,
    team: row.team,
    community: row.community
}})
""")
source.execute(f"""
UNWIND {cypher_list(edge_rows, ['src', 'dst', 'strength'])} AS row
MATCH (a:Person {{id: row.src}}), (b:Person {{id: row.dst}})
CREATE (a)-[:KNOWS {{strength: row.strength}}]->(b)
""")
source.query_as_arrow("""
MATCH (p:Person)
RETURN p.id AS id, p.name AS name, p.team AS team, p.community AS community
ORDER BY p.id
""", 4096).get_as_arrow().to_pandas()



,id,name,team,community
0,0,Ada,A,0
1,1,Ben,A,0
2,2,Cora,A,0
3,3,Drew,A,0
4,4,Eli,A,0
5,5,Fay,B,0
6,6,Gus,B,0
7,7,Ivy,B,0
8,8,Jin,B,0
9,9,Kai,B,0


## Understanding Modularity (Q)

Modularity measures how much more densely connected nodes are within communities than would be expected in a random network with the same degree distribution.

**High modularity (Q ≳ 0.3–0.7):** Many more edges fall within communities than expected by chance. Communities are relatively well separated.

**Q ≈ 0:** The partition is no better than random with respect to community structure.

**Q < 0:** Fewer within-community edges than expected by chance; the partition is worse than random.

## [NOT REQUIRED; DEMONSTRATION PURPOSES ONLY] Export the database, close it, and import it into a fresh in-memory instance

The export writes Parquet files plus `schema.cypher` into a temporary directory. The source instance is then closed before the next in-memory instance imports that Parquet export.

In [3]:
if os.path.exists(export_dir) and os.path.isdir(export_dir):
    shutil.rmtree(export_dir)

source.execute(f"EXPORT DATABASE '{export_dir}'")
source.close()

print(sorted(p.name for p in export_dir.iterdir()))

source_db = ladybug.Database()
source = ladybug.Connection(source_db)
source.execute(f"IMPORT DATABASE '{export_dir}'")

source.query_as_arrow("""
MATCH (p:Person)
RETURN p.id AS id, p.name AS name, p.team AS team, p.community AS community
ORDER BY p.id
""", 4096).get_as_arrow().to_pandas()


['KNOWS_Person_Person.parquet', 'Person.parquet', 'copy.cypher', 'index.cypher', 'schema.cypher']


,id,name,team,community
0,0,Ada,A,0
1,1,Ben,A,0
2,2,Cora,A,0
3,3,Drew,A,0
4,4,Eli,A,0
5,5,Fay,B,0
6,6,Gus,B,0
7,7,Ivy,B,0
8,8,Jin,B,0
9,9,Kai,B,0


## Extract CSR from the imported database and run Icebug Leiden

`query_as_arrow(...).csr()` turns the imported rowid projection into CSR arrays, and those arrays feed `Graph.fromCSR` directly.

In [4]:
n_nodes = source.query_as_arrow(
    "MATCH (p:Person) RETURN count(p) AS n",
    1024,
).get_as_arrow().column(0)[0].as_py()

relationship_result = source.query_as_arrow("""
MATCH (a:Person)-[r:KNOWS]->(b:Person)
RETURN a.rowid AS src_rowid,
       r.rowid AS rel_rowid,
       b.rowid AS dst_rowid
ORDER BY src_rowid, rel_rowid
""", 4096)
relationship_csr = relationship_result.csr()

indptr = relationship_csr.indptr.cast(pa.uint64())
indices = relationship_csr.indices.cast(pa.uint64())

graph = ib.Graph.fromCSR(n_nodes, False, indices, indptr)
leiden = ib.community.ParallelLeidenView(graph, iterations=4, gamma=1.0, randomize=False)
leiden.run()
partition = leiden.getPartition()
modularity = ib.community.Modularity().getQuality(partition, graph)

community_frame = source.query_as_arrow("""
MATCH (p:Person)
RETURN p.id AS id, p.name AS name, p.team AS team
ORDER BY p.id
""", 4096).get_as_arrow().to_pandas()
community_frame["community"] = [partition.subsetOf(i) for i in range(graph.numberOfNodes())]

print(f"Icebug found {community_frame['community'].nunique()} communities; modularity Q={modularity:.4f}")
community_frame


Icebug found 4 communities; modularity Q=0.6801


,id,name,team,community
0,0,Ada,A,0
1,1,Ben,A,0
2,2,Cora,A,0
3,3,Drew,A,0
4,4,Eli,A,0
5,5,Fay,B,1
6,6,Gus,B,1
7,7,Ivy,B,1
8,8,Jin,B,1
9,9,Kai,B,1


In [5]:
# Visualize the graph with Leiden communities
from icebug import vizbridges

# Interactive widget with nodes colored by community assignment
vizbridges.widgetFromGraph(graph, nodePartition=partition)

CytoscapeWidget(cytoscape_layout={'name': 'cose'}, cytoscape_style=[{'selector': 'node', 'css': {'background-c…

## Assign the Icebug communities back to the nodes

The imported `Person` table already has a `community` column, so Cypher `SET` can write the Leiden partition back onto each node.

In [6]:
for node_id, community in community_frame[["id", "community"]].itertuples(index=False):
    source.execute(f"MATCH (p:Person {{id: {int(node_id)}}}) SET p.community = {int(community)}")

updated_nodes = source.query_as_arrow("""
MATCH (p:Person)
RETURN p.id AS id, p.name AS name, p.team AS team, p.community AS community
ORDER BY p.community, p.team, p.id
""", 4096).get_as_arrow().to_pandas()

updated_nodes

,id,name,team,community
0,0,Ada,A,0
1,1,Ben,A,0
2,2,Cora,A,0
3,3,Drew,A,0
4,4,Eli,A,0
5,5,Fay,B,1
6,6,Gus,B,1
7,7,Ivy,B,1
8,8,Jin,B,1
9,9,Kai,B,1


In [7]:
source.query_as_arrow("""
MATCH (a:Person)-[r:KNOWS]->(b:Person)
RETURN a.name AS source,
       a.community AS source_community,
       b.name AS target,
       b.community AS target_community,
       r.strength AS strength
ORDER BY source_community, source, target
LIMIT 12
""", 4096).get_as_arrow().to_pandas()


,source,source_community,target,target_community,strength
0,Ada,0,Ben,0,10
1,Ada,0,Cora,0,10
2,Ada,0,Drew,0,10
3,Ada,0,Eli,0,10
4,Ada,0,Fay,1,1
5,Ben,0,Ada,0,10
6,Ben,0,Cora,0,10
7,Ben,0,Drew,0,10
8,Ben,0,Eli,0,10
9,Cora,0,Ada,0,10


## Clean up temporary notebook data

Remove the exported Parquet directory and close the imported instance once the results have been displayed.

In [8]:
source.close()

if os.path.exists(workdir) and os.path.isdir(workdir):
    shutil.rmtree(workdir)

print("temporary data removed:", not workdir.exists())

temporary data removed: True
